# 🎓 Leson 2 : Antrene pwòp modèl IA oryantasyon pa w

Nan Leson 1, nou te kreye yon ti jeu de données : **enterè elèv la → metye → fòmasyon**.
Jodi a, nou pral **ajiste (fine-tune)** yon modèl pre-antrene pou l ka li sa yon elèv ekri epi rekòmande l yon domèn.

**Sa n ap fè etap pa etap :**
1. 🔧 Enstale zouti yo
2. 📚 Konprann kijan fine-tuning mache
3. 🗂️ Prepare done yo nan bon fòma
4. 🚀 Antrene premye modèl nou an
5. 🧪 Teste konseye oryantasyon an
6. 💾 Sove epi rechaje modèl la

> ⚙️ **Anvan w kòmanse :** nan Colab, ale nan *Runtime → Change runtime type* epi chwazi **GPU (T4)**. Antrennman an ap pran 2 a 5 minit.

## 🔧 Etap 1 : Enstale zouti yo

- **transformers** : bibliyotèk Hugging Face pou modèl IA yo
- **datasets** : pou jere done antrennman yo
- **accelerate** : pou antrennman an mache vit sou GPU

In [ ]:
!pip install -q transformers datasets accelerate

import numpy as np
import pandas as pd
import torch
import json

print("✅ Zouti yo enstale !")
print("GPU disponib :", torch.cuda.is_available())

## 📚 Etap 2 : Kisa fine-tuning ye ?

Imajine yon elèv ki deja konn li ak ekri byen (**modèl pre-antrene a**). Li pa konn anyen sou oryantasyon pwofesyonèl.
Nou ba li kèk egzanp (**done nou yo**), epi li aprann rekonèt ki domèn ki koresponn ak chak kalite enterè.

Nou pral fè sa yo rele **klasifikasyon tèks** :

| Antre (sa elèv la ekri) | ➜ | Sòti (domèn modèl la chwazi) |
|---|---|---|
| « Mwen renmen soude sikui elektwonik » | ➜ | Elektwonik ak sistèm anbake |

Chak domèn gen deja yon lis metye ak fòmasyon (nou te prepare yo nan Leson 1).

**Modèl nou chwazi :** `distilbert-base-multilingual-cased` — yon modèl lejè ki konprann plis pase 100 lang, sa ki pèmèt nou melanje fransè ak kreyòl.

## 🗂️ Etap 3 : Prepare done yo

### 3.1 — Tablo domèn yo (soti nan Leson 1)
Nou kenbe menm 14 domèn yo, ak metye ak fòmasyon yo.

> ⚠️ **Fòmatè a dwe verifye** non etablisman yo epi ajoute fòmasyon ki egziste nan zòn li.

In [ ]:
domen = [
    {"non": "Devlopman lojisyèl ak web",
     "metye": "Développeur logiciel, Développeur web, Ingénieur full-stack",
     "fomasyon": "ESIH (informatique),INUKA,  Université Quisqueya, formations en ligne (freeCodeCamp, OpenClassrooms)"},
    {"non": "Done ak matematik",
     "metye": "Data scientist, Statisticien, Analyste financier",
     "fomasyon": "UEH (Faculté des Sciences), ESIH,CTPEA, cours en ligne (Coursera, edX)"},
    {"non": "Sikoloji ak travay sosyal",
     "metye": "Psychologue, Conseiller, Travailleur social",
     "fomasyon": "UEH (Sciences Humaines, Ethnologie), Université Quisqueya"},
    {"non": "Atizay ak design",
     "metye": "Graphiste, Designer UI/UX, Illustrateur",
     "fomasyon": "Formations en ligne en design (Domestika, Coursera) et création d'un portfolio"},
    {"non": "Biyoloji ak sante",
     "metye": "Chercheur biomédical, Médecin, Pharmacien",
     "fomasyon": "UEH (Faculté de Médecine et de Pharmacie),Quiskeya, UNDH, USFD, formations en sciences de la santé"},
    {"non": "Ekriti ak jounalis",
     "metye": "Rédacteur, Journaliste, Auteur",
     "fomasyon": "Formations en communication et journalisme, ateliers d'écriture"},
    {"non": "Jeni mekanik ak sivil",
     "metye": "Ingénieur mécanicien, Ingénieur civil, Architecte",
     "fomasyon": "UEH (Faculté des Sciences),GOC, CANADO Technique, Haiti Tech,CETEMO, écoles d'ingénieurs"},
    {"non": "Biznis ak antreprenarya",
     "metye": "Analyste d'affaires, Entrepreneur, Responsable marketing",
     "fomasyon": "UEH (INAGHEI),ENST, Université Quisqueya"},
    {"non": "Ansèyman",
     "metye": "Enseignant, Professeur, Créateur de contenus pédagogiques",
     "fomasyon": "UEH (École Normale Supérieure)"},
    {"non": "Mizik ak son",
     "metye": "Producteur de musique, Ingénieur du son, Compositeur",
     "fomasyon": "Écoles de musique, formations en ligne en production musicale, CHCL(UEH), ENART(UEH)"},
    {"non": "Robotik",
     "metye": "Ingénieur en robotique, Ingénieur en systèmes embarqués, Technicien en automatisation",
     "fomasyon": "ESIH (infotronique), UEH (Faculté des Sciences), ROS 2 et robotique en ligne"},
    {"non": "Elektwonik ak sistèm anbake",
     "metye": "Ingénieur en électronique, Développeur de systèmes embarqués, Technicien IoT",
     "fomasyon": "ESIH (infotronique),GOC, formations en électronique et microcontrôleurs"},
    {"non": "Entèlijans atifisyèl",
     "metye": "Ingénieur en intelligence artificielle, Ingénieur en apprentissage automatique, Chercheur en IA",
     "fomasyon": "ESIH,UNIQ, cours en ligne d'IA (Hugging Face, Coursera, fast.ai)"},
    {"non": "Teknoloji pou sante",
     "metye": "Ingénieur biomédical, Développeur de logiciels de santé, Informaticien de santé",
     "fomasyon": "ESIH, UEH (Faculté de Médecine et de Pharmacie), projets de santé numérique"},
]

id2label = {i: d["non"] for i, d in enumerate(domen)}
label2id = {v: k for k, v in id2label.items()}
print(f"✅ {len(domen)} domèn prepare")

### 3.2 — Ogmante done yo (data augmentation)

Pwoblèm : nan Leson 1, nou te gen **yon sèl fraz pa domèn**. Se twò piti ! Yon modèl pa ka aprann ak yon sèl egzanp.

Solisyon : nou ekri **plizyè fason diferan** pou di menm bagay — an fransè **ak** an kreyòl.
Konsa modèl la aprann sans lan, pa sèlman mo yo.

In [ ]:
egzanp = {
    0: ["J'adore programmer et créer des sites web",
        "Je passe mes soirées à coder des applications",
        "J'aime comprendre comment fonctionne une application mobile",
        "Créer un jeu vidéo ou un site me passionne",
        "Mwen renmen pwograme ak kreye sit entènèt",
        "Mwen pase tout tan m ap ekri kòd sou òdinatè",
        "Mwen ta renmen kreye yon aplikasyon pou telefòn"],
    1: ["J'aime résoudre des problèmes de maths et travailler avec des données",
        "Les statistiques et les graphiques me fascinent",
        "J'aime analyser des chiffres pour trouver des tendances",
        "Calculer des probabilités, c'est mon passe-temps",
        "Mwen renmen rezoud pwoblèm matematik",
        "Mwen renmen travay ak chif ak estatistik",
        "Mwen fò nan matematik e m renmen analize done"],
    2: ["J'aime aider les gens et comprendre la psychologie",
        "J'écoute souvent les problèmes de mes amis et je les conseille",
        "Je veux comprendre pourquoi les gens agissent comme ils le font",
        "Le bien-être des familles de mon quartier me tient à cœur",
        "Mwen renmen ede moun ki gen pwoblèm",
        "Mwen renmen koute moun e ba yo konsèy",
        "Mwen vle konprann kijan moun panse"],
    3: ["Je suis passionné par l'art et le design créatif",
        "Je dessine tout le temps dans mes cahiers",
        "J'aime créer des logos et des affiches",
        "J'aime imaginer l'apparence d'une application",
        "Mwen renmen desine ak penn",
        "Mwen renmen kreye logo ak afich",
        "Koulè ak bèl desen se pasyon m"],
    4: ["J'adore la biologie et je veux aider à guérir les maladies",
        "Le corps humain et les médicaments me passionnent",
        "Je veux soigner les malades de ma communauté",
        "J'aime les expériences de laboratoire en SVT",
        "Mwen renmen biyoloji e m vle geri moun",
        "Mwen vle vin doktè pou ede moun malad",
        "Mwen renmen etidye kò moun ak medikaman"],
    5: ["J'aime écrire des histoires et créer du contenu",
        "Je tiens un journal et j'écris des poèmes",
        "J'aime faire des reportages sur ce qui se passe autour de moi",
        "La radio et l'actualité m'intéressent beaucoup",
        "Mwen renmen ekri istwa",
        "Mwen renmen fè repòtaj sou sa k ap pase nan zòn mwen",
        "Mwen ekri powèm ak tèks pou rezo sosyal"],
    6: ["J'aime construire des choses et comprendre comment fonctionnent les machines",
        "Je démonte les moteurs pour voir comment ils marchent",
        "J'aimerais construire des ponts et des maisons solides",
        "Les plans de bâtiments me fascinent",
        "Mwen renmen konstwi bagay ak men m",
        "Mwen renmen demonte motè pou wè kijan yo mache",
        "Mwen ta renmen konstwi kay ki kanpe fèm devan tranblemanntè"],
    7: ["Je m'intéresse aux affaires et à l'entrepreneuriat",
        "Je vends déjà des produits à l'école",
        "J'aimerais créer ma propre entreprise",
        "Le marketing et la publicité m'intéressent",
        "Mwen renmen fè biznis",
        "Mwen vle louvri pwòp antrepriz pa m",
        "Mwen renmen vann e jere lajan"],
    8: ["J'adore enseigner et expliquer des notions aux autres",
        "J'aide souvent mes camarades à faire leurs devoirs",
        "J'aime préparer des exposés pour la classe",
        "Transmettre ce que je sais me rend heureux",
        "Mwen renmen anseye",
        "Mwen renmen esplike leson bay zanmi m",
        "Mwen ta renmen vin pwofesè lekòl"],
    9: ["J'aime la musique et je veux créer des bandes sonores",
        "Je joue de la guitare et je compose des chansons",
        "J'aime enregistrer et mixer des sons",
        "Le konpa et les instruments me passionnent",
        "Mwen renmen mizik",
        "Mwen jwe gita e m konpoze chante",
        "Mwen ta renmen fè beat ak pwodiksyon mizik"],
    10: ["Je suis fasciné par les robots et par leur capacité à se déplacer seuls",
         "J'aimerais construire un robot qui nettoie la maison",
         "Les drones et les véhicules autonomes m'intéressent",
         "J'aime l'automatisation des usines",
         "Mwen renmen robo",
         "Mwen ta renmen konstwi yon robo ki ka mache pou kont li",
         "Dwòn ak machin otonòm enterese m anpil"],
    11: ["J'aime souder des circuits et programmer des microcontrôleurs",
         "Je bricole avec des Arduino et des capteurs",
         "Je répare les téléphones et les radios",
         "Connecter des objets à internet me passionne",
         "Mwen renmen soude sikui elektwonik",
         "Mwen renmen pwograme Arduino ak kaptè",
         "Mwen repare telefòn ak radyo nan katye a"],
    12: ["Je veux que les machines apprennent et prennent des décisions",
         "Les chatbots et l'intelligence artificielle me passionnent",
         "J'aimerais créer un programme qui reconnaît les images",
         "L'apprentissage automatique m'intrigue",
         "Mwen renmen entèlijans atifisyèl",
         "Mwen vle fè machin aprann pou kont yo",
         "Mwen ta renmen kreye yon chatbot ki pale kreyòl"],
    13: ["Je veux utiliser la technologie pour améliorer la santé en Haïti",
         "J'aimerais créer une application pour suivre les patients",
         "Je veux aider les hôpitaux avec l'informatique",
         "Concevoir des appareils médicaux m'intéresse",
         "Mwen vle itilize teknoloji pou amelyore sante an Ayiti",
         "Mwen ta renmen kreye yon aplikasyon pou swiv pasyan",
         "Mwen vle ede lopital yo ak enfòmatik"],
}

teks, etikèt = [], []
for label, fraz_yo in egzanp.items():
    for f in fraz_yo:
        teks.append(f)
        etikèt.append(label)

df = pd.DataFrame({"text": teks, "label": etikèt})
df["domen"] = df["label"].map(id2label)
print(f"✅ Nou gen {len(df)} egzanp kounye a (olye de 14) !")
df.sample(8, random_state=1)

### 3.3 — Separe done yo : antrennman ak tès

Nou sere **20 %** egzanp yo sou kote. Modèl la p ap janm wè yo pandan antrennman an.
Apre sa, n ap itilize yo pou n wè si li vrèman aprann, oswa si li jis aprann pa kè.

> 🧠 Se tankou yon egzamen : si w bay elèv la menm kesyon li te etidye a, ou pa konnen si l konprann.

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

df_antre, df_tes = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)

ds_antre = Dataset.from_pandas(df_antre[["text", "label"]].reset_index(drop=True))
ds_tes = Dataset.from_pandas(df_tes[["text", "label"]].reset_index(drop=True))

print(f"📘 Antrennman : {len(ds_antre)} egzanp")
print(f"📝 Tès        : {len(ds_tes)} egzanp")

### 3.4 — Tokenizasyon : transfòme tèks an nimewo

Ou sonje sa nan Leson 1 ? Modèl la pa li lèt, li li **nimewo**. Tokenizer a koupe chak fraz an moso (tokens) epi ba chak moso yon nimewo.

In [ ]:
from transformers import AutoTokenizer

NON_MODEL = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(NON_MODEL)

def tokenize(lo):
    return tokenizer(lo["text"], truncation=True, max_length=64)

ds_antre = ds_antre.map(tokenize, batched=True)
ds_tes = ds_tes.map(tokenize, batched=True)

# Gade kijan yon fraz kreyòl koupe
egz = "Mwen renmen soude sikui elektwonik"
print("Fraz  :", egz)
print("Tokens:", tokenizer.tokenize(egz))

## 🚀 Etap 4 : Antrene modèl la

### 4.1 — Chaje modèl pre-antrene a
Nou di modèl la gen **14 sòti posib** (14 domèn yo). Dènye kouch li a nouvo — se li ki pral aprann.

> Li nòmal si w wè yon mesaj ki di gen kèk pwa ki « newly initialized ». Se jisteman kouch nou pral antrene a.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    NON_MODEL,
    num_labels=len(domen),
    id2label=id2label,
    label2id=label2id,
)
print("✅ Modèl chaje :", NON_MODEL)

### 4.2 — Konfigire antrennman an

| Paramèt | Valè | Sa sa vle di |
|---|---|---|
| `num_train_epochs` | 20 | Modèl la ap wè tout done yo 20 fwa |
| `learning_rate` | 5e-5 | Vitès li aprann (twò vit = li pèdi pye, twò dousman = li pa aprann) |
| `per_device_train_batch_size` | 8 | Kantite fraz li wè an menm tan |

Nou bezwen anpil *epochs* paske done nou yo piti anpil.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def kalkile_metrik(rezilta):
    logits, vre_label = rezilta
    prediksyon = np.argmax(logits, axis=-1)
    return {"presizyon": (prediksyon == vre_label).mean()}

paramet = TrainingArguments(
    output_dir="antrennman_oryantasyon",
    num_train_epochs=20,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=paramet,
    train_dataset=ds_antre,
    eval_dataset=ds_tes,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=kalkile_metrik,
)
print("✅ Tout bagay pare pou antrennman an")

### 4.3 — Lanse antrennman an 🚀
Gade kolòn **Validation Loss** (dwe desann) ak **Presizyon** (dwe monte).

In [ ]:
trainer.train()

In [ ]:
rezilta = trainer.evaluate()
print(f"🎯 Presizyon sou done tès yo : {rezilta['eval_presizyon']:.0%}")

> 💡 **Kijan pou n li rezilta a ?** Ak sèlman ~20 fraz tès, chak erè fè presizyon an desann anpil.
> Yon rezilta ant 60 % ak 90 % nòmal pou yon jeu de données tou piti konsa.
> Si w jwenn 100 %, pa twò kontan : tès la twò piti pou n konnen vrèman.

## 🧪 Etap 5 : Teste konseye oryantasyon an

Fonksyon `konseye()` la bay **3 domèn ki pi pwobab** yo, ak metye ak fòmasyon pou chak.
Nou bay 3 rekòmandasyon paske yon moun ka enterese nan plizyè domèn an menm tan (egzanp : robotik **ak** elektwonik).

In [ ]:
def konseye(teks, top_k=3):
    model.eval()
    antre = tokenizer(teks, return_tensors="pt", truncation=True, max_length=64).to(model.device)
    with torch.no_grad():
        logits = model(**antre).logits
    pwobabilite = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    meyo = np.argsort(pwobabilite)[::-1][:top_k]

    print(f"🧑‍🎓 Elèv la di : « {teks} »\n")
    for rang, i in enumerate(meyo, start=1):
        d = domen[i]
        print(f"{rang}. {d['non']}  ({pwobabilite[i]:.0%})")
        print(f"   💼 Metye    : {d['metye']}")
        print(f"   🎓 Fòmasyon : {d['fomasyon']}\n")

konseye("Mwen renmen fè eksperyans ak kaptè epi pwograme yon ti robo")

In [ ]:
# Eseye ak fraz modèl la PA JANM wè
fraz_nouvo = [
    "Je veux devenir infirmière pour aider les enfants",
    "Mwen renmen kalkile e fè tablo ak chif",
    "Mwen ta renmen kreye yon aplikasyon ki ede doktè yo",
    "J'aime chanter à l'église et jouer du piano",
]
for f in fraz_nouvo:
    konseye(f)
    print("-" * 60)

### 🔍 Obsève limit modèl la
Eseye fraz sa yo, epi reflechi : poukisa modèl la ezite oswa twonpe l ?
- Yon fraz ki pa gen okenn rapò ak travay : *« Mwen renmen manje diri ak pwa »*
- Yon domèn ki pa nan lis la : *« Mwen renmen plante pye bwa ak elve bèt »* (agrikilti)

👉 Modèl la **toujou** bay yon repons, menm lè li pa konnen. Se poutèt sa yon konseye oryantasyon moun rete enpòtan !

In [ ]:
konseye("Mwen renmen plante pye bwa ak elve bèt")

## 💾 Etap 6 : Sove epi rechaje modèl la

Nou sove modèl la, tokenizer a **ak** tablo domèn yo (pou nou ka toujou jwenn metye ak fòmasyon yo).

In [ ]:
DOSYE = "modèl_oryantasyon"
trainer.save_model(DOSYE)
tokenizer.save_pretrained(DOSYE)
with open(f"{DOSYE}/domen.json", "w", encoding="utf-8") as f:
    json.dump(domen, f, ensure_ascii=False, indent=2)

print(f"💾 Modèl la sove nan dosye « {DOSYE} »")

In [ ]:
# Rechaje modèl la (tankou si nou te fèmen Colab epi n tounen)
from transformers import pipeline

klasifye = pipeline("text-classification", model=DOSYE, top_k=3)
with open(f"{DOSYE}/domen.json", encoding="utf-8") as f:
    domen_rechaje = {d["non"]: d for d in json.load(f)}

rezilta = klasifye("Je rêve de construire des drones")
if isinstance(rezilta[0], list):
    rezilta = rezilta[0]
for r in rezilta:
    print(f"{r['label']} ({r['score']:.0%}) → {domen_rechaje[r['label']]['metye']}")

In [ ]:
# (Opsyonèl) Telechaje modèl la sou òdinatè w
!zip -rq modèl_oryantasyon.zip modèl_oryantasyon
from google.colab import files
files.download("modèl_oryantasyon.zip")

## 🎯 Mini-defi n° 2

**MISYON OU :** ajoute yon **15yèm domèn** — pa egzanp **Agrikilti ak anviwònman**.

1. Ajoute yon nouvo eleman nan lis `domen` (non, metye, fòmasyon).
2. Ajoute omwen **6 fraz** (fransè ak kreyòl) nan `egzanp` ak nimewo `14`.
3. Relanse tout selil yo depi etap 3 (*Runtime → Run all*).
4. Teste : `konseye("Mwen renmen plante pye bwa ak elve bèt")` — èske repons lan chanje ?

> 💡 Panse ak sa k egziste vrèman an Ayiti : FAMV (UEH), Ministè Agrikilti, pwojè reyabilitasyon anviwònman…

## 🎓 Leson an fini !

**SA OU APRANN JODI A :**
- ✅ Kisa fine-tuning ye (adapte yon modèl ki deja konn lang)
- ✅ Poukisa nou bezwen **ogmante** done yo
- ✅ Separe done antrennman ak done tès
- ✅ Antrene yon modèl klasifikasyon ak `Trainer`
- ✅ Bay 3 rekòmandasyon ak pwobabilite yo
- ✅ Sove epi rechaje yon modèl pèsonalize

**⚖️ RESPONSABILITE :** yon modèl konsa se yon **zouti pou ede reflechi**, pa yon desizyon final.
Li ka repete prejije ki nan done yo (pa egzanp, si tout egzanp « doktè » yo te ekri pa gason). Toujou verifye ak yon konseye moun.

**DEVWA (OPSYONÈL) :**
1. Fini mini-defi n° 2.
2. Mande 5 zanmi ekri yon fraz sou sa yo renmen, teste yo, epi note konbyen fwa modèl la gen rezon.
3. Bonis : chanje `NON_MODEL` pou `xlm-roberta-base` (pi gwo, pi bon pou lang ki pa gen anpil done tankou kreyòl). Konpare presizyon an.